In [1]:
"""
================================================================================
RAMD-Net : Rotor-Aware Micro-Doppler Network  --  complete PyTorch pipeline
================================================================================
ONE file: model + leak-free split + training (multi-seed stats) + SNR-robustness
+ ablation. Everything below the model is standard PyTorch; sklearn/scipy are
used ONLY for evaluation metrics and significance tests, never for training.

WHY THE SPLIT IS RECORDING-AWARE (this is the whole point):
DIAT-uSAT spectrograms are sliding-window frames of continuous recordings, so
figureN and figureN+1 are near-duplicates (measured: ~86% of consecutive frames
correlate >0.95). A RANDOM frame split scatters near-duplicates across train and
test, leaking the test set and inflating accuracy to a saturated ~98% where no
architectural component can be distinguished. This pipeline therefore splits so
that train/val/test never share a recording (whole-recording holdout), or -- if
frames are numbered without detectable gaps -- splits each class chronologically
with a buffer so no near-duplicate straddles the boundary. There is NO random
split option, on purpose.

THE MODEL (RAMD-Net) -- physics-grounded, replaces the earlier CSPDNet design:
A micro-Doppler spectrogram has two physically distinct axes. The Anisotropic
Dual-Axis (ADA) block processes them separately: a (k x 1) conv along the
Doppler/frequency axis (blade-tip velocity, spectral extent) and a (1 x k) conv
along the slow-time axis (periodicity / HERM flash rate), fused and reweighted by
a dual-axis gate. Depth is added by SEQUENTIALLY stacking ADA blocks within a
resolution stage (the `blocks` tuple), not by a multi-scale head -- an earlier
version averaged three parallel classification heads at three resolutions, which
was inherited unflagged from the original DIAT-(Light)CSPDNet paper's
DecoupledHead-style design and is dropped here: the component ablation showed it
indistinguishable from a single head (90.56+/-4.15 vs 91.34+/-3.27 at 20 epochs,
mirroring the original paper's own 97.54 vs 97.90 finding), and keeping it would
have left an unflagged inherited component sitting next to the part actually being
claimed as novel. Default ~75K params / 0.29MB / ~54M FLOPs (vs DIAT-RadSATNet
453K / 2.21MB / 590M).

HONEST NOVELTY SCOPE -- none of the ADA block's three sub-mechanisms is new in
isolation: the (k x 1)/(1 x k) asymmetric factorization is Inception-v3's
decomposition (Szegedy et al.), the residual path is ResNet, and the gate's
squeeze-MLP-sigmoid structure is Squeeze-and-Excitation. What is being claimed is
narrower: (1) the PHYSICAL motivation for why an asymmetric split is the right
inductive bias for THIS signal -- the two axes are not interchangeable spatial
dimensions but distinct physical quantities -- where Inception-v3 used the same
factorization purely for FLOP savings on natural images with no such claim, and
(2) the gate is conditioned on a contrast between two physically distinct pooled
descriptors (Doppler-response energy vs time-response energy), not a single
generic descriptor. The actual evidence for the claim is empirical: the SNR
stress ablation shows the gap between full RAMD-Net and the isotropic-conv
control widens as SNR drops (93.2->17.5 vs 85.8->16.6), consistent with "this
inductive bias matters under degraded conditions" rather than "more parameters
happened to help."

INFLUENCES to cite (NOT CSPDNet): Inception-v3 (asymmetric n x 1 / 1 x n convs),
ResNet (residual paths), Squeeze-and-Excitation (gating, here modified into a
dual-axis contrast gate); differentiate from axis-decoupled micro-Doppler HAR
networks (we derive the decoupling from rotor physics for SUAV classification).

MODES (set MODE at the bottom):
  "train"               -> multi-seed training + mean/std/CI + report (leak-free)
  "stress"              -> SNR-robustness ablation (5 key variants) <-- shows novelty
  "ablation_components" -> component ablation on the leak-free split
  "ablation_depth_width"-> depth then width sweep
  "split_only"          -> just build + inspect the split, no training
================================================================================
"""
import os, re, json, time, random, math, hashlib
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision import transforms
from PIL import Image

from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix, classification_report)
from scipy import stats as sps

# =============================================================== CONFIG ========
CONFIG = dict(
    data_dir   = "/kaggle/input/datasets/kaushikar/drone-usat/DIAT-uSAT_dataset",  # <- set this
    out_dir    = "./ramdnet_out",
    img_size   = 224,
    batch_size = 32,
    epochs     = 100,
    lr         = 1e-3,
    min_lr     = 1e-5,
    weight_decay = 1e-2,
    early_stop_patience = 15,
    split_seed = 42,
    model_seeds = [0, 1, 2, 3, 4],
    num_workers = 4,
    # recording-aware split knobs
    split_cache     = "recording_data_split.json",
    gap_threshold   = 1,      # frame-index gap > this => new recording
    min_recordings  = 3,      # >= this many recordings/class => whole-recording holdout
    boundary_buffer = 5,      # frames dropped at each chronological boundary
    # architecture
    channels   = (12, 24, 32, 40),
    kernel     = 7,
    blocks     = (1, 1, 1, 1),
    block_type = "ada",       # "ada" or "isotropic"
    pool_type  = "avg",
    use_residual = True,
    use_doppler = True,
    use_time    = True,
    use_spectral = False,     # 3rd branch: spectral-envelope (off by default;
                              # the branch-combination ablation toggles it)
    use_cadence  = False,     # 4th branch: cadence/FFT (off by default)
    use_gate    = True,
    fusion      = "sum",        # block-design knob (Ablation 1): "sum" | "concat"
    gate_style  = "contrast",   # block-design knob (Ablation 1): "contrast" | "se" | "none"
    multi_scale = False,      # dropped: see module docstring -- inherited unflagged
                              # from the original paper's head design, ablation
                              # showed it indistinguishable from single-scale
    # ablation / stress
    ablation_seeds = [0, 1, 2],
    ablation_epochs = 100,         # full convergence for the block-design and
                                    # multiplicity ablations (was 20, then 50)
    ablation_early_stop_patience = 18,   # derived from logs: patience=12 cut some
                                          # variants at 38-42 still improving; main
                                          # run at patience=15 maxed at 51. 18 gives
                                          # late-improvers room, stops land ~30-55.
    stress_seeds = [0, 1],
    snr_list_db  = [None, 20, 15, 10, 5, 0],   # None = clean
)

MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]


def normalize_class_name(raw):
    s = raw.strip().lower().replace("-", " ").replace("_", " ")
    s = " ".join(s.split()); p = s.split()
    if p and p[-1] in {"1", "2", "3"}: s = " ".join(p[:-1]).strip()
    if "long blade" in s:  return "3_long_blade_rotor"
    if "short blade" in s: return "3_short_blade_rotor"
    if "mini helicopter" in s or "minihelicopter" in s or "helicopter" in s:
        return "Bird+mini-helicopter"
    if s == "bird" or s.endswith(" bird") or s == "bionic bird": return "Bird"
    if "rc plane" in s or s.startswith("rc"): return "RC_plane"
    if "drone" in s or "quad" in s: return "drone"
    return raw


def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def _std(a):
    """Sample std that returns 0.0 for n<=1 instead of nan (quiet single-seed runs)."""
    a = np.asarray(a, dtype=float)
    return float(a.std(ddof=1)) if a.size > 1 else 0.0



def _frame_index(path):
    m = re.findall(r"(\d+)", os.path.basename(path))
    return int(m[-1]) if m else None


# =============================================================== DATASET =======
class MicroDopplerDataset(Dataset):
    IMG_EXT = (".png", ".jpg", ".jpeg", ".bmp")

    def __init__(self, samples, class_to_idx, transform=None):
        self.samples = samples; self.class_to_idx = class_to_idx; self.transform = transform

    def __len__(self): return len(self.samples)

    def __getitem__(self, i):
        path, label = self.samples[i]
        img = Image.open(path).convert("RGB")
        if self.transform: img = self.transform(img)
        return img, label

    @staticmethod
    def scan(data_dir):
        """Returns samples as (path, merged_class_idx). Also returns
        raw_group[i] = the ORIGINAL sub-folder name for sample i (e.g.
        '3_short_blade_rotor_1' vs '_2') -- needed because two raw folders
        that merge into the same class can each restart their own figureN
        numbering from 1, which would corrupt frame-gap recording detection
        if the merged class were sorted by index alone."""
        data_dir = Path(data_dir); raw = []
        for sub in sorted(p for p in data_dir.iterdir() if p.is_dir()):
            cls = normalize_class_name(sub.name)
            for f in sorted(sub.rglob("*")):       # sorted => reproducible scan order
                if f.suffix.lower() in MicroDopplerDataset.IMG_EXT:
                    raw.append((str(f), cls, sub.name))
        classes = sorted({c for _, c, _ in raw})
        c2i = {c: i for i, c in enumerate(classes)}
        samples = [(p, c2i[c]) for p, c, _ in raw]
        raw_group = [g for _, _, g in raw]
        return samples, classes, c2i, raw_group


# ===================================================== RECORDING-AWARE SPLIT ===
def detect_recordings(samples, classes, raw_group, gap_threshold=1):
    """Group into recordings using (merged_class, raw_subfolder) as the key so
    that two raw folders merged into one class (e.g. *_1 and *_2, each with
    their own figureN numbering starting at 1) are never sorted together --
    that interleaving was destroying gap detection and silently corrupting the
    split. Within each (class, raw_subfolder) group, contiguous frame-index
    runs (gap <= gap_threshold) are one recording."""
    by_key = defaultdict(list)   # (class, raw_subfolder) -> [(frame_idx, sample_idx)]
    for si, ((path, lab), grp) in enumerate(zip(samples, raw_group)):
        by_key[(classes[lab], grp)].append((_frame_index(path), si))

    recordings = defaultdict(list)   # class -> [[sample_idx,...], ...]
    print("Recording structure (by frame-index gaps, per raw sub-folder):")
    per_class_groups = defaultdict(list)
    for (cls, grp), items in by_key.items():
        per_class_groups[cls].append(grp)
        items = sorted(items, key=lambda t: (t[0] is None, t[0]))
        recs, cur, prev = [], [], None
        for fidx, si in items:
            if fidx is None:
                if cur: recs.append(cur); cur = []
                recs.append([si]); prev = None; continue
            if prev is not None and fidx - prev > gap_threshold:
                recs.append(cur); cur = []
            cur.append(si); prev = fidx
        if cur: recs.append(cur)
        recordings[cls].extend(recs)

    for cls in classes:
        recs = recordings[cls]
        sizes = [len(r) for r in recs]
        groups = sorted(set(per_class_groups[cls]))
        print(f"  {cls:24s}: {len(recs):3d} recording(s) across raw folder(s) "
              f"{groups}, sizes min/med/max = {min(sizes)}/{int(np.median(sizes))}/{max(sizes)}")
    return dict(recordings)


def _split_one_class(recs, target_va, target_te, buffer):
    """recs: list of recordings for ONE class, each a list of sample indices
    in chronological order within that recording.

    - 1 recording: no structural boundary signal at all -> chronological cut
      with a frame buffer at each cut point (flagged explicitly).
    - >=2 recordings: process SMALLEST-FIRST. Each recording is used to fill
      whichever target (test, then val) still needs frames. If a recording is
      BIGGER than what's still needed, only the needed amount is carved off
      (from one end, with a buffer dropped before the rest) -- the leftover
      of that same recording goes to train, instead of dumping the entire
      recording into test/val and wildly overshooting the 15% target (which
      starves train). A recording fully consumed by a target with nothing
      left over needs no buffer at all -- it's separated from other splits
      by a real recording boundary, not a synthetic cut.
    """
    if len(recs) == 1:
        order = recs[0]; n = len(order)
        i_tr, i_va = int((1 - target_va - target_te) * n), int((1 - target_te) * n)
        tr = order[: max(0, i_tr - buffer)]
        va = order[i_tr + buffer: max(i_tr + buffer, i_va - buffer)]
        te = order[i_va + buffer:]
        return tr, va, te, "chronological_buffer_NO_BOUNDARY_SIGNAL"

    total = sum(len(r) for r in recs)
    te_target, va_target = target_te * total, target_va * total
    te_n = va_n = 0
    te, va, tr = [], [], []
    n_partial = 0

    for r in sorted(recs, key=len):   # smallest first: don't waste a big block on a small need
        remaining = r[:]
        if te_n < te_target and remaining:
            need = te_target - te_n
            if len(remaining) <= need:
                te += remaining; te_n += len(remaining); remaining = []
            else:
                take = max(1, int(round(need)))
                te += remaining[-take:]; te_n += take       # carve from the end
                remaining = remaining[:-(take + buffer)] if len(remaining) > take + buffer else []
                n_partial += 1
        if va_n < va_target and remaining:                  # same recording's LEFTOVER can still fill val
            need = va_target - va_n
            if len(remaining) <= need:
                va += remaining; va_n += len(remaining); remaining = []
            else:
                take = max(1, int(round(need)))
                va += remaining[:take]; va_n += take         # carve from the start
                remaining = remaining[take + buffer:] if len(remaining) > take + buffer else []
                n_partial += 1
        tr += remaining

    method = "whole_recording_holdout" + (f" ({n_partial} partial carve(s))" if n_partial else "")
    return tr, va, te, method


def get_split(cfg, samples, labels, classes, raw_group):
    cache = os.path.join(cfg["out_dir"], cfg["split_cache"])
    os.makedirs(cfg["out_dir"], exist_ok=True)
    if os.path.exists(cache):
        with open(cache) as f: saved = json.load(f)
        if saved.get("n_total") == len(samples):
            p2i = {p: i for i, (p, _) in enumerate(samples)}
            try:
                tr = [p2i[p] for p in saved["train_paths"]]
                va = [p2i[p] for p in saved["val_paths"]]
                te = [p2i[p] for p in saved["test_paths"]]
                print(f"Loaded split <- {cache} (train={len(tr)} val={len(va)} "
                      f"test={len(te)})")
                for cls, m in saved.get("per_class_method", {}).items():
                    print(f"  {cls:24s}: {m}")
                return tr, va, te
            except KeyError:
                pass

    recordings = detect_recordings(samples, classes, raw_group, cfg["gap_threshold"])
    B = cfg["boundary_buffer"]
    tr, va, te, per_class_method = [], [], [], {}
    print(f"\nPer-class split (whole-recording packing where boundaries exist, "
          f"buffered chronological cut only where they don't):")
    for cls in classes:
        c_tr, c_va, c_te, method = _split_one_class(recordings[cls], 0.15, 0.15, B)
        tr += c_tr; va += c_va; te += c_te
        per_class_method[cls] = f"{method} ({len(recordings[cls])} recording(s))"
        print(f"  {cls:24s}: {method}  (train={len(c_tr)} val={len(c_va)} test={len(c_te)})")

    flagged = [c for c, m in per_class_method.items() if "NO_BOUNDARY_SIGNAL" in m]
    if flagged:
        print(f"\n  NOTE: {len(flagged)} class(es) had no detectable recording "
              f"boundary ({flagged}) -- their split relies on a {B}-frame "
              f"chronological buffer only. Disclose as a limitation.")

    method_summary = ("whole_recording_holdout" if not flagged else
                      "mixed (see per_class_method)")
    payload = dict(n_total=len(samples), method=method_summary, split_seed=cfg["split_seed"],
                   per_class_method=per_class_method,
                   train_paths=[samples[i][0] for i in tr],
                   val_paths=[samples[i][0] for i in va],
                   test_paths=[samples[i][0] for i in te])
    with open(cache, "w") as f: json.dump(payload, f)
    print(f"\nSaved split -> {cache} (train={len(tr)} val={len(va)} test={len(te)}, "
          f"method={method_summary})")
    return tr, va, te


def _loader(samples, c2i, tf, cfg, shuffle):
    return DataLoader(MicroDopplerDataset(samples, c2i, tf), batch_size=cfg["batch_size"],
                      shuffle=shuffle, num_workers=cfg["num_workers"], pin_memory=True)


def build_loaders(cfg):
    samples, classes, c2i, raw_group = MicroDopplerDataset.scan(cfg["data_dir"])
    labels = [l for _, l in samples]
    print(f"Found {len(samples)} images across {len(classes)} classes: {classes}")
    for c, i in c2i.items(): print(f"  {c:24s}: {labels.count(i)}")
    tr, va, te = get_split(cfg, samples, labels, classes, raw_group)
    sub = lambda ids: [samples[i] for i in ids]
    norm = transforms.Normalize(MEAN, STD)
    train_tf = transforms.Compose([
        transforms.Resize((cfg["img_size"], cfg["img_size"])),
        transforms.RandomHorizontalFlip(),        # time reversal; NOT vertical (Doppler sign)
        transforms.ColorJitter(brightness=0.1), transforms.ToTensor(), norm])
    eval_tf = transforms.Compose([
        transforms.Resize((cfg["img_size"], cfg["img_size"])), transforms.ToTensor(), norm])
    return (_loader(sub(tr), c2i, train_tf, cfg, True),
            _loader(sub(va), c2i, eval_tf, cfg, False),
            _loader(sub(te), c2i, eval_tf, cfg, False), classes, sub(te), c2i)


# =============================================================== MODEL =========
class ConvBNAct(nn.Module):
    def __init__(s, ci, co, k=3, st=1, p=None, act=True):
        super().__init__()
        p = (k - 1) // 2 if p is None else p
        s.c = nn.Conv2d(ci, co, k, st, p, bias=False); s.bn = nn.BatchNorm2d(co)
        s.a = nn.SiLU(inplace=True) if act else nn.Identity()
    def forward(s, x): return s.a(s.bn(s.c(x)))


class ADABlock(nn.Module):
    """Multi-branch decoupled block. Each branch extracts a PHYSICALLY DISTINCT
    micro-Doppler cue with a DIFFERENT operator on a DIFFERENT representation, so
    the branches share no computation ('no common ground'):

      doppler  : (k x 1) 2D conv along the frequency axis -> blade-tip velocity,
                 local spectral structure.
      time     : (1 x k) 2D conv along the slow-time axis -> HERM flash, local
                 periodicity texture.
      spectral : collapses TIME (mean over W) into a per-frequency profile, then
                 a 1D conv OVER FREQUENCY -> overall Doppler-bandwidth shape,
                 independent of time.
      cadence  : magnitude FFT ALONG TIME, averaged over frequency rows, then a
                 1D conv over the cadence axis -> rotation rate as a spectral
                 peak, in a transformed (frequency-of-time) domain, not pixels.

    Branches are fused (sum or concat+project), a gate conditioned on branch
    descriptors reweights channels, then pointwise mix + residual. Each branch is
    independently toggleable; block-level design knobs (fusion, gate_style) are
    swept whole, never by dissecting the decoupling itself."""
    def __init__(s, c, k=7, r=8, use_doppler=True, use_time=True,
                 use_spectral=False, use_cadence=False,
                 use_gate=True, use_residual=True, pool_type="avg",
                 fusion="sum", gate_style="contrast"):
        super().__init__()
        assert use_doppler or use_time or use_spectral or use_cadence, "need >=1 branch"
        assert fusion in ("sum", "concat"), fusion
        assert gate_style in ("contrast", "se", "none"), gate_style
        s.use_doppler, s.use_time = use_doppler, use_time
        s.use_spectral, s.use_cadence = use_spectral, use_cadence
        s.use_residual = use_residual
        s.fusion = fusion
        # gate_style="none" is equivalent to use_gate=False; honor either
        s.gate_style = "none" if not use_gate else gate_style
        s.use_gate = s.gate_style != "none"
        s.pool = F.adaptive_avg_pool2d if pool_type == "avg" else F.adaptive_max_pool2d
        if use_doppler:
            s.dop = nn.Sequential(nn.Conv2d(c, c, (k, 1), 1, ((k - 1) // 2, 0), bias=False),
                                  nn.BatchNorm2d(c), nn.SiLU(inplace=True))
        if use_time:
            s.tim = nn.Sequential(nn.Conv2d(c, c, (1, k), 1, (0, (k - 1) // 2), bias=False),
                                  nn.BatchNorm2d(c), nn.SiLU(inplace=True))
        if use_spectral:
            s.spec_conv = nn.Conv1d(c, c, k, padding=(k - 1) // 2, bias=False)
            s.spec_bn = nn.BatchNorm1d(c); s.spec_act = nn.SiLU(inplace=True)
        if use_cadence:
            s.cad_conv = nn.Conv1d(c, c, k, padding=(k - 1) // 2, bias=False)
            s.cad_bn = nn.BatchNorm1d(c); s.cad_act = nn.SiLU(inplace=True)
        nb = int(use_doppler) + int(use_time) + int(use_spectral) + int(use_cadence)
        s.nb = nb
        # fusion: "sum" adds branch features (all c channels); "concat" stacks
        # them (c*nb channels) then projects back to c with a 1x1.
        if fusion == "concat":
            s.fuse_proj = nn.Sequential(nn.Conv2d(c * nb, c, 1, bias=False), nn.BatchNorm2d(c))
        # gate: "contrast" conditions on ALL branch descriptors concatenated
        # (sees the contrast between physically-distinct branches); "se" is a
        # plain single-descriptor squeeze-excite on the fused map.
        if s.gate_style == "contrast":
            s.gate = nn.Sequential(nn.Linear(c * nb, max(c // r, 4)), nn.SiLU(inplace=True),
                                   nn.Linear(max(c // r, 4), c), nn.Sigmoid())
        elif s.gate_style == "se":
            s.gate = nn.Sequential(nn.Linear(c, max(c // r, 4)), nn.SiLU(inplace=True),
                                   nn.Linear(max(c // r, 4), c), nn.Sigmoid())
        s.pw = nn.Sequential(nn.Conv2d(c, c, 1, bias=False), nn.BatchNorm2d(c))
        s.act = nn.SiLU(inplace=True)

    def _spectral(s, x):
        prof = x.mean(dim=3)                                # (B,C,H) collapse time
        prof = s.spec_act(s.spec_bn(s.spec_conv(prof)))     # 1D conv over frequency
        return prof.unsqueeze(3).expand(-1, -1, -1, x.shape[3])

    def _cadence(s, x):
        spec = torch.fft.rfft(x.float(), dim=3).abs()       # FFT along time
        cad = spec.mean(dim=2)                              # (B,C,F) collapse freq rows
        cad = s.cad_act(s.cad_bn(s.cad_conv(cad)))          # 1D conv over cadence
        cad = cad.mean(dim=2, keepdim=True).unsqueeze(3)    # (B,C,1,1)
        return cad.expand(-1, -1, x.shape[2], x.shape[3])

    def forward(s, x):
        feats, descs = [], []
        if s.use_doppler:
            d = s.dop(x); feats.append(d); descs.append(s.pool(d, 1).flatten(1))
        if s.use_time:
            t = s.tim(x); feats.append(t); descs.append(s.pool(t, 1).flatten(1))
        if s.use_spectral:
            sp = s._spectral(x); feats.append(sp); descs.append(s.pool(sp, 1).flatten(1))
        if s.use_cadence:
            cd = s._cadence(x); feats.append(cd); descs.append(s.pool(cd, 1).flatten(1))
        if s.fusion == "concat" and len(feats) > 1:
            fused = s.fuse_proj(torch.cat(feats, dim=1))
        else:
            fused = sum(feats) if len(feats) > 1 else feats[0]
        if s.use_gate:
            if s.gate_style == "contrast":
                w = s.gate(torch.cat(descs, 1))
            else:  # se: squeeze the fused map itself
                w = s.gate(s.pool(fused, 1).flatten(1))
            fused = fused * w.unsqueeze(-1).unsqueeze(-1)
        out = s.pw(fused)
        return s.act(out + x) if s.use_residual else s.act(out)


class IsotropicBlock(nn.Module):
    """Ablation control: ordinary k x k conv block (matched channels/residual)."""
    def __init__(s, c, k=7, use_residual=True, pool_type="avg", **_):
        super().__init__()
        s.use_residual = use_residual
        s.conv = nn.Sequential(nn.Conv2d(c, c, k, 1, (k - 1) // 2, bias=False),
                               nn.BatchNorm2d(c), nn.SiLU(inplace=True))
        s.pw = nn.Sequential(nn.Conv2d(c, c, 1, bias=False), nn.BatchNorm2d(c))
        s.act = nn.SiLU(inplace=True)
    def forward(s, x):
        out = s.pw(s.conv(x))
        return s.act(out + x) if s.use_residual else s.act(out)


def make_block(bt, c, k, use_doppler, use_time, use_gate, use_residual, pool_type,
               use_spectral=False, use_cadence=False, fusion="sum", gate_style="contrast"):
    if bt == "isotropic":
        return IsotropicBlock(c, k=k, use_residual=use_residual, pool_type=pool_type)
    return ADABlock(c, k=k, use_doppler=use_doppler, use_time=use_time,
                    use_spectral=use_spectral, use_cadence=use_cadence,
                    use_gate=use_gate, use_residual=use_residual, pool_type=pool_type,
                    fusion=fusion, gate_style=gate_style)


class RAMDNet(nn.Module):
    def __init__(s, num_classes=6, in_ch=3, ch=(12, 24, 32, 40), k=7, blocks=(1, 1, 1, 1),
                 block_type="ada", pool_type="avg", use_doppler=True, use_time=True,
                 use_spectral=False, use_cadence=False,
                 use_gate=True, use_residual=True, multi_scale=True,
                 fusion="sum", gate_style="contrast"):
        super().__init__()
        s.multi_scale = multi_scale
        bk = dict(k=k, use_doppler=use_doppler, use_time=use_time, use_gate=use_gate,
                  use_spectral=use_spectral, use_cadence=use_cadence,
                  use_residual=use_residual, pool_type=pool_type,
                  fusion=fusion, gate_style=gate_style)
        stage = lambda c, n: nn.Sequential(*[make_block(block_type, c, **bk) for _ in range(n)])
        s.stem = nn.Sequential(ConvBNAct(in_ch, ch[0] // 2, 3, 2), ConvBNAct(ch[0] // 2, ch[0], 3, 2))
        s.stage0 = stage(ch[0], blocks[0])
        s.down1 = ConvBNAct(ch[0], ch[1], 3, 2); s.stage1 = stage(ch[1], blocks[1])
        s.down2 = ConvBNAct(ch[1], ch[2], 3, 2); s.stage2 = stage(ch[2], blocks[2])
        s.down3 = ConvBNAct(ch[2], ch[3], 3, 2); s.stage3 = stage(ch[3], blocks[3])
        if multi_scale:
            s.heads = nn.ModuleList([nn.Linear(c, num_classes) for c in ch[1:]])
        else:
            s.head = nn.Linear(ch[3], num_classes)
    def forward(s, x):
        x = s.stem(x); x = s.stage0(x)
        x = s.down1(x); p1 = s.stage1(x)
        x = s.down2(p1); p2 = s.stage2(x)
        x = s.down3(p2); p3 = s.stage3(x)
        if s.multi_scale:
            return sum(h(F.adaptive_avg_pool2d(f, 1).flatten(1))
                       for f, h in zip([p1, p2, p3], s.heads)) / 3
        return s.head(F.adaptive_avg_pool2d(p3, 1).flatten(1))


def build_model(cfg, num_classes):
    return RAMDNet(num_classes=num_classes, ch=tuple(cfg["channels"]), k=cfg["kernel"],
                   blocks=tuple(cfg["blocks"]), block_type=cfg["block_type"],
                   pool_type=cfg["pool_type"], use_doppler=cfg["use_doppler"],
                   use_time=cfg["use_time"], use_gate=cfg["use_gate"],
                   use_spectral=cfg.get("use_spectral", False),
                   use_cadence=cfg.get("use_cadence", False),
                   fusion=cfg.get("fusion", "sum"), gate_style=cfg.get("gate_style", "contrast"),
                   use_residual=cfg["use_residual"], multi_scale=cfg["multi_scale"])


# =========================================================== COMPLEXITY ========
def _macs_fallback(model, img_size=224):
    """Count MACs via forward hooks. Pure shape arithmetic -> always on CPU,
    on a detached copy, so it never interacts with the caller's device."""
    macs = [0]; hooks = []
    def ch(m, i, o):
        n, co, h, w = o.shape; kh, kw = m.kernel_size
        macs[0] += n * co * h * w * (m.in_channels // m.groups) * kh * kw
    def lh(m, i, o): macs[0] += o.shape[0] * m.in_features * m.out_features
    for m in model.modules():
        if isinstance(m, nn.Conv2d): hooks.append(m.register_forward_hook(ch))
        elif isinstance(m, nn.Linear): hooks.append(m.register_forward_hook(lh))
    was = model.training; model.eval()
    with torch.no_grad(): model(torch.randn(1, 3, img_size, img_size))
    for h in hooks: h.remove()
    model.train(was); return macs[0]


def complexity(model, img_size=224, device="cpu"):
    """Params / size / MACs / FLOPs. Counting is done on CPU regardless of
    `device` (it is just shape arithmetic), so it works whether or not thop is
    installed and never causes a CPU/CUDA tensor mismatch."""
    out = {"params": sum(p.numel() for p in model.parameters())}
    out["size_mb"] = (sum(p.numel() * p.element_size() for p in model.parameters())
                      + sum(b.numel() * b.element_size() for b in model.buffers())) / 1024**2
    model_cpu = model.to("cpu")
    try:
        from thop import profile
        macs, _ = profile(model_cpu, inputs=(torch.randn(1, 3, img_size, img_size),), verbose=False)
        out["MACs_M"], out["FLOPs_M"], out["flop_method"] = macs / 1e6, 2 * macs / 1e6, "thop"
    except Exception as e:
        macs = _macs_fallback(model_cpu, img_size)
        out["MACs_M"], out["FLOPs_M"], out["flop_method"] = macs / 1e6, 2 * macs / 1e6, f"fallback ({type(e).__name__})"
    return out


@torch.no_grad()
def benchmark_latency(model, img_size=224, device="cpu", warmup=20, iters=200):
    model = model.to(device).eval(); x = torch.randn(1, 3, img_size, img_size, device=device)
    for _ in range(warmup): model(x)
    if device.startswith("cuda"): torch.cuda.synchronize()
    ts = []
    for _ in range(iters):
        t0 = time.perf_counter(); model(x)
        if device.startswith("cuda"): torch.cuda.synchronize()
        ts.append((time.perf_counter() - t0) * 1000)
    ts = np.array(ts)
    return dict(latency_ms_mean=float(ts.mean()), latency_ms_std=float(ts.std()),
                fps=float(1000 / ts.mean()))


# =============================================================== TRAIN =========
def run_epoch(model, loader, crit, opt, device, train):
    model.train() if train else model.eval()
    tot = correct = 0; loss_sum = 0.0
    torch.set_grad_enabled(train)
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        if train: opt.zero_grad()
        out = model(x); loss = crit(out, y)
        if train: loss.backward(); opt.step()
        loss_sum += loss.item() * x.size(0)
        correct += (out.argmax(1) == y).sum().item(); tot += x.size(0)
    torch.set_grad_enabled(True)
    return loss_sum / tot, correct / tot


@torch.no_grad()
def predict(model, loader, device):
    model.eval(); P, Y = [], []
    for x, y in loader:
        P.append(model(x.to(device)).argmax(1).cpu().numpy()); Y.append(y.numpy())
    return np.concatenate(P), np.concatenate(Y)


def train_single(cfg, loaders, num_classes, seed, device):
    set_seed(seed)
    tr, va, te = loaders[0], loaders[1], loaders[2]
    model = build_model(cfg, num_classes).to(device)
    crit = nn.CrossEntropyLoss()
    opt = AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    sch = CosineAnnealingLR(opt, T_max=cfg["epochs"], eta_min=cfg["min_lr"])
    best, best_state, bad = 0.0, None, 0
    for ep in range(1, cfg["epochs"] + 1):
        run_epoch(model, tr, crit, opt, device, True)
        _, vacc = run_epoch(model, va, crit, opt, device, False); sch.step()
        if vacc > best:
            best, best_state, bad = vacc, {k: v.cpu().clone() for k, v in model.state_dict().items()}, 0
        else:
            bad += 1
            if cfg["early_stop_patience"] and bad >= cfg["early_stop_patience"]:
                print(f"    early stop @ epoch {ep} (best val {best:.4f})"); break
        if ep % 10 == 0 or ep == 1:
            print(f"    ep{ep:3d} val_acc={vacc:.4f} best={best:.4f}")
    model.load_state_dict(best_state)
    P, Y = predict(model, te, device)
    pr, rc, f1, _ = precision_recall_fscore_support(Y, P, average="weighted", zero_division=0)
    return dict(seed=seed, test_acc=accuracy_score(Y, P), precision=pr, recall=rc, f1=f1,
                preds=P.tolist(), labels=Y.tolist(), best_val=best, state=best_state)


# ========================================================= STATS / SIG =========
def bootstrap_ci(labels, preds, n=10000, alpha=0.05, seed=0):
    rng = np.random.default_rng(seed); labels, preds = np.asarray(labels), np.asarray(preds)
    accs = np.array([(preds[idx] == labels[idx]).mean()
                     for idx in (rng.integers(0, len(labels), len(labels)) for _ in range(n))])
    return float(np.percentile(accs, 100 * alpha / 2)), float(np.percentile(accs, 100 * (1 - alpha / 2)))


def mcnemar(labels, pa, pb):
    labels = np.asarray(labels); a = np.asarray(pa) == labels; b = np.asarray(pb) == labels
    n01, n10 = int(np.sum(a & ~b)), int(np.sum(~a & b)); n = n01 + n10
    if n == 0: return dict(n01=n01, n10=n10, p_value=1.0, test="none")
    if n < 25:
        p = min(2 * sps.binom.cdf(min(n01, n10), n, 0.5), 1.0)
        return dict(n01=n01, n10=n10, p_value=float(p), test="exact")
    chi2 = (abs(n01 - n10) - 1) ** 2 / n
    return dict(n01=n01, n10=n10, statistic=float(chi2), p_value=float(sps.chi2.sf(chi2, 1)), test="chi2_cc")


def aggregate(results):
    accs = np.array([r["test_acc"] for r in results]); f1s = np.array([r["f1"] for r in results])
    m, s = accs.mean(), _std(accs)
    if len(accs) > 1:
        tc = sps.t.ppf(0.975, len(accs) - 1); half = tc * s / math.sqrt(len(accs)); ci = (m - half, m + half)
    else:
        ci = (m, m)
    best = sorted(results, key=lambda r: r["test_acc"])[len(results) // 2]
    bci = bootstrap_ci(best["labels"], best["preds"])
    return dict(acc_mean=float(m), acc_std=float(s), acc_ci95=[float(ci[0]), float(ci[1])],
                f1_mean=float(f1s.mean()), f1_std=_std(f1s),
                bootstrap_ci95=[bci[0], bci[1]], n_runs=len(accs))


# =============================================================== SNR ===========
def add_awgn(x01, snr_db):
    if snr_db is None: return x01
    sig = x01.pow(2).mean(dim=(-1, -2, -3), keepdim=True)
    return (x01 + torch.randn_like(x01) * (sig / 10 ** (snr_db / 10)).sqrt()).clamp(0, 1)


@torch.no_grad()
def evaluate_under_snr(model, test_samples, c2i, cfg, snr_list, device, seed=0):
    model = model.to(device).eval()
    raw_tf = transforms.Compose([transforms.Resize((cfg["img_size"], cfg["img_size"])), transforms.ToTensor()])
    loader = DataLoader(MicroDopplerDataset(test_samples, c2i, raw_tf), batch_size=cfg["batch_size"],
                        shuffle=False, num_workers=cfg["num_workers"], pin_memory=True)
    mean = torch.tensor(MEAN, device=device).view(1, 3, 1, 1); std = torch.tensor(STD, device=device).view(1, 3, 1, 1)
    out = {}
    for snr in snr_list:
        torch.manual_seed(seed); correct = total = 0
        for x01, y in loader:
            x01, y = x01.to(device), y.to(device)
            pred = model((add_awgn(x01, snr) - mean) / std).argmax(1)
            correct += (pred == y).sum().item(); total += y.size(0)
        out["clean" if snr is None else f"{snr}dB"] = correct / total
    return out


# =============================================================== RUNS ==========
def run_experiment(cfg, loaders=None):
    """Main multi-seed training run, leak-free split. Checkpoints PER SEED
    (metrics as JSON + that seed's weights as a small .pth -- the model is
    only ~0.3MB so 5 checkpoints cost ~1.5MB total) so a kill mid-run loses
    at most the seed that was in progress, not the whole run. On restart,
    completed seeds are loaded from disk and skipped."""
    device = "cuda" if torch.cuda.is_available() else "cpu"
    os.makedirs(cfg["out_dir"], exist_ok=True); print(f"Device: {device}")
    if loaders is None:
        loaders = build_loaders(cfg)
    num_classes = len(loaders[3])
    comp = complexity(build_model(cfg, num_classes), cfg["img_size"], device)
    lat = benchmark_latency(build_model(cfg, num_classes), cfg["img_size"], device)
    print("Complexity:", {k: round(v, 4) if isinstance(v, float) else v for k, v in comp.items()})
    print("Latency   :", {k: round(v, 4) for k, v in lat.items()})

    results = []
    for sd in cfg["model_seeds"]:
        meta_path = os.path.join(cfg["out_dir"], f"train_seed{sd}_meta.json")
        ckpt_path = os.path.join(cfg["out_dir"], f"train_seed{sd}.pth")
        if os.path.exists(meta_path) and os.path.exists(ckpt_path):
            with open(meta_path) as f: meta = json.load(f)
            print(f"\n=== seed {sd} [skipped -- loaded from checkpoint] "
                  f"test_acc={meta['test_acc']:.4f} ===")
            results.append(meta)
            continue
        print(f"\n=== seed {sd} ===")
        r = train_single(cfg, loaders, num_classes, sd, device)
        print(f"    -> test_acc={r['test_acc']:.4f} f1={r['f1']:.4f}")
        torch.save({"state_dict": r["state"]}, ckpt_path)
        meta = {k: v for k, v in r.items() if k != "state"}
        with open(meta_path, "w") as f: json.dump(meta, f)
        results.append(meta)

    agg = aggregate(results)
    print("\n================ summary (leak-free split) ================")
    print(f"Accuracy: {agg['acc_mean']*100:.2f} +/- {agg['acc_std']*100:.2f} %  "
          f"(95% CI [{agg['acc_ci95'][0]*100:.2f}, {agg['acc_ci95'][1]*100:.2f}])")
    print(f"Bootstrap 95% CI (median seed): [{agg['bootstrap_ci95'][0]*100:.2f}, "
          f"{agg['bootstrap_ci95'][1]*100:.2f}] %")
    best = sorted(results, key=lambda r: r["test_acc"])[len(results) // 2]
    print("\n" + classification_report(best["labels"], best["preds"],
          target_names=loaders[3], digits=4, zero_division=0))
    np.savez(os.path.join(cfg["out_dir"], "ramdnet_preds.npz"),
             labels=np.array(best["labels"]), preds=np.array(best["preds"]))
    best_state = torch.load(os.path.join(cfg["out_dir"], f"train_seed{best['seed']}.pth"),
                            weights_only=True)["state_dict"]
    torch.save({"state_dict": best_state, "classes": loaders[3], "config": cfg},
               os.path.join(cfg["out_dir"], "ramdnet_best.pth"))
    with open(os.path.join(cfg["out_dir"], "ramdnet_summary.json"), "w") as f:
        json.dump(dict(config=cfg, complexity=comp, latency=lat, aggregate=agg,
                       confusion_matrix=confusion_matrix(best["labels"], best["preds"]).tolist(),
                       classes=loaders[3]), f, indent=2)
    print(f"\nSaved -> {cfg['out_dir']}/ramdnet_summary.json")
    return results, loaders


KEY_VARIANTS = {
    "RAMD-Net (full)":      dict(),
    "No dual-axis gate":    dict(use_gate=False),
    "Doppler-only branch":  dict(use_time=False),
    "Time-only branch":     dict(use_doppler=False),
    "Isotropic conv (kxk)": dict(block_type="isotropic"),
}


def run_stress_ablation(cfg, loaders=None):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    if loaders is None:
        loaders = build_loaders(cfg)
    num_classes = len(loaders[3])
    test_samples, c2i = loaders[4], loaders[5]
    snr_list = cfg["snr_list_db"]; seeds = cfg["stress_seeds"]
    cols = ["clean" if s is None else f"{s}dB" for s in snr_list]
    out_path = os.path.join(cfg["out_dir"], "stress_ablation.json")
    results = json.load(open(out_path)) if os.path.exists(out_path) else {}
    if results: print(f"Resuming: {len(results)} variant(s) already done.")

    print("\n" + "=" * 88)
    print("SNR-ROBUSTNESS ABLATION (recording-aware, leak-free split)")
    print(f"({len(KEY_VARIANTS)} variants x {len(seeds)} seeds x {len(snr_list)} SNR levels, "
          f"{cfg['ablation_epochs']} epochs each)")
    print("=" * 88)
    for name, ov in KEY_VARIANTS.items():
        if name in results: print(f"{name:22s} [skipped]"); continue
        c = dict(cfg); c.update(ov)
        c["epochs"] = cfg["ablation_epochs"]; c["early_stop_patience"] = cfg["ablation_early_stop_patience"]
        comp = complexity(build_model(c, num_classes), c["img_size"])
        per_seed = []
        for sd in seeds:
            r = train_single(c, loaders, num_classes, sd, device)
            m = build_model(c, num_classes).to(device); m.load_state_dict(r["state"])
            per_seed.append(evaluate_under_snr(m, test_samples, c2i, c, snr_list, device, seed=sd))
        agg = {col: float(np.mean([ps[col] for ps in per_seed])) for col in cols}
        results[name] = dict(mean=agg, params=comp["params"], FLOPs_M=round(comp["FLOPs_M"], 2), n_seeds=len(seeds))
        json.dump(results, open(out_path, "w"), indent=2)
        print(f"{name:22s} " + "  ".join(f"{agg[c_]*100:5.1f}" for c_ in cols) + f"   params={comp['params']:,}")

    print("\n" + f"{'Variant':22s} " + "  ".join(f"{c_:>5s}" for c_ in cols))
    print("-" * 88)
    for name in KEY_VARIANTS:
        if name in results:
            m = results[name]["mean"]
            print(f"{name:22s} " + "  ".join(f"{m[c_]*100:5.1f}" for c_ in cols))
    print(f"\nSaved -> {out_path}")
    print("If 'full' ties others at clean but stays higher as SNR drops -> components matter.")
    return results


COMPONENT_VARIANTS = {
    "RAMD-Net (full, single-scale)": dict(),    # new baseline: single head, sequential ADA depth
    "No dual-axis gate": dict(use_gate=False),
    "Doppler-only branch": dict(use_time=False), "Time-only branch": dict(use_doppler=False),
    "No residual connection": dict(use_residual=False), "Isotropic conv (kxk)": dict(block_type="isotropic"),
    "Multi-scale head (3-way avg)": dict(multi_scale=True),   # inverse check: re-confirm dropping it was right
    "Max pooling": dict(pool_type="max"),
    "Kernel k=3": dict(kernel=3), "Kernel k=5": dict(kernel=5), "Kernel k=9": dict(kernel=9),
}


def run_component_ablation(cfg, loaders=None):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    if loaders is None: loaders = build_loaders(cfg)
    num_classes = len(loaders[3]); seeds = cfg["ablation_seeds"]
    out_path = os.path.join(cfg["out_dir"], "ablation_components.json")
    rows = json.load(open(out_path)) if os.path.exists(out_path) else []
    done = {r["variant"] for r in rows}
    if done: print(f"Resuming: {len(done)} variant(s) done.")
    print(f"\nABLATION (components) on leak-free split: {len(COMPONENT_VARIANTS)} x {len(seeds)} seeds, "
          f"{cfg['ablation_epochs']} epochs")
    for name, ov in COMPONENT_VARIANTS.items():
        if name in done: print(f"{name:24s} [skipped]"); continue
        c = dict(cfg); c.update(ov)
        c["epochs"] = cfg["ablation_epochs"]; c["early_stop_patience"] = cfg["ablation_early_stop_patience"]
        comp = complexity(build_model(c, num_classes), c["img_size"])
        accs = [train_single(c, loaders, num_classes, sd, device)["test_acc"] for sd in seeds]
        accs = np.array(accs)
        rows.append(dict(variant=name, acc_mean=float(accs.mean()),
                         acc_std=_std(accs),
                         params=comp["params"], FLOPs_M=round(comp["FLOPs_M"], 2)))
        json.dump(rows, open(out_path, "w"), indent=2)
        print(f"{name:24s} acc={accs.mean()*100:6.2f}+/-{_std(accs)*100:4.2f}  params={comp['params']:,}")
    print(f"Saved -> {out_path}")
    return rows


# ============================ BRANCH-COMBINATION ABLATION (stage-1 search) =====
# Decide WHICH of the 4 decoupled branches to keep, BEFORE any depth/width sweep.
# Ranked on VALIDATION accuracy -- the test set stays sealed until the winning
# combination is chosen, so this two-stage (combination -> depth/width) search
# stays honest. We don't enumerate all 15 non-empty subsets; we test the
# physically motivated ladder: each axis alone, the two conv axes, the two conv
# axes + each new branch, and the full four. d=Doppler t=time s=spectral c=cadence.
COMBO_VARIANTS = {
    "d (Doppler only)":        dict(use_doppler=True,  use_time=False, use_spectral=False, use_cadence=False),
    "t (time only)":           dict(use_doppler=False, use_time=True,  use_spectral=False, use_cadence=False),
    "d+t (current 2-branch)":  dict(use_doppler=True,  use_time=True,  use_spectral=False, use_cadence=False),
    "d+t+s (+spectral)":       dict(use_doppler=True,  use_time=True,  use_spectral=True,  use_cadence=False),
    "d+t+c (+cadence)":        dict(use_doppler=True,  use_time=True,  use_spectral=False, use_cadence=True),
    "d+t+s+c (full 4-branch)": dict(use_doppler=True,  use_time=True,  use_spectral=True,  use_cadence=True),
    "s+c (new branches only)": dict(use_doppler=False, use_time=False, use_spectral=True,  use_cadence=True),
}


def run_combo_ablation(cfg, loaders=None):
    """Stage-1 of the coarse-to-fine search: which branch COMBINATION is best.
    Selection metric is VALIDATION accuracy (test recorded but NOT used to pick,
    so the later depth/width sweep on the winner doesn't leak the test set)."""
    device = "cuda" if torch.cuda.is_available() else "cpu"
    if loaders is None: loaders = build_loaders(cfg)
    num_classes = len(loaders[3]); seeds = cfg["ablation_seeds"]
    out_path = os.path.join(cfg["out_dir"], "ablation_combo.json")
    rows = json.load(open(out_path)) if os.path.exists(out_path) else []
    done = {r["variant"] for r in rows}
    if done: print(f"Resuming: {len(done)} combination(s) done.")
    print(f"\nBRANCH-COMBINATION ABLATION (stage 1, ranked on VALIDATION): "
          f"{len(COMBO_VARIANTS)} combos x {len(seeds)} seeds, {cfg['ablation_epochs']} epochs")
    for name, ov in COMBO_VARIANTS.items():
        if name in done: print(f"{name:26s} [skipped]"); continue
        c = dict(cfg); c.update(ov)
        c["epochs"] = cfg["ablation_epochs"]; c["early_stop_patience"] = cfg["ablation_early_stop_patience"]
        comp = complexity(build_model(c, num_classes), c["img_size"])
        val_accs, test_accs = [], []
        for sd in seeds:
            r = train_single(c, loaders, num_classes, sd, device)
            val_accs.append(r["best_val"]); test_accs.append(r["test_acc"])
        val_accs, test_accs = np.array(val_accs), np.array(test_accs)
        rows.append(dict(variant=name,
                         val_mean=float(val_accs.mean()), val_std=_std(val_accs),
                         test_mean=float(test_accs.mean()), test_std=_std(test_accs),
                         params=comp["params"], FLOPs_M=round(comp["FLOPs_M"], 2)))
        json.dump(rows, open(out_path, "w"), indent=2)
        print(f"{name:26s} VAL={val_accs.mean()*100:6.2f}+/-{_std(val_accs)*100:4.2f}  "
              f"(test={test_accs.mean()*100:5.2f})  params={comp['params']:,}")
    # rank on validation, report winner
    rows_sorted = sorted(rows, key=lambda r: -r["val_mean"])
    print("\n  Ranked by VALIDATION accuracy (selection metric):")
    for r in rows_sorted:
        print(f"    {r['variant']:26s} val={r['val_mean']*100:6.2f}  "
              f"params={r['params']:>7,d}")
    print(f"\n  -> Stage-1 winner (by val): {rows_sorted[0]['variant']}")
    print(f"     Run depth/width sweep on THIS combination next; test stays sealed.")
    print(f"Saved -> {out_path}")
    return rows


# ====================== ABLATION 1: BLOCK-DESIGN VARIANTS (val-ranked) =========
# Treat the ADA block as a coherent UNIT (axis-decoupling FIXED -- never dissected
# into doppler-only/time-only here). Sweep whole-block design choices a reader
# would consider as real alternatives: fusion style, gate style, strip-kernel
# length. Each row is a complete, sensible block. Ranked on VALIDATION; the
# spread between rows comes from genuine design differences, not hand-tuning.
# `base_combo` is applied first (the stage-1 branch winner) so this runs on the
# chosen branch set, not an arbitrary one.
BLOCK_DESIGN_VARIANTS = {
    "sum + contrast-gate, k7":   dict(fusion="sum",    gate_style="contrast", kernel=7),
    "sum + contrast-gate, k5":   dict(fusion="sum",    gate_style="contrast", kernel=5),
    "sum + contrast-gate, k9":   dict(fusion="sum",    gate_style="contrast", kernel=9),
    "concat + contrast-gate, k7": dict(fusion="concat", gate_style="contrast", kernel=7),
    "sum + SE-gate, k7":         dict(fusion="sum",    gate_style="se",       kernel=7),
    "sum + no-gate, k7":         dict(fusion="sum",    gate_style="none",     kernel=7),
    "concat + SE-gate, k7":      dict(fusion="concat", gate_style="se",       kernel=7),
    "concat + contrast-gate, k9": dict(fusion="concat", gate_style="contrast", kernel=9),
}


def _val_ablation(cfg, variants, out_name, title, base_combo=None, loaders=None):
    """Shared val-ranked ablation runner: trains each variant for ablation_seeds,
    records val (selection) + test (sealed) means, ranks by val, names winner."""
    device = "cuda" if torch.cuda.is_available() else "cpu"
    if loaders is None: loaders = build_loaders(cfg)
    num_classes = len(loaders[3]); seeds = cfg["ablation_seeds"]
    out_path = os.path.join(cfg["out_dir"], out_name)
    rows = json.load(open(out_path)) if os.path.exists(out_path) else []
    done = {r["variant"] for r in rows}
    if done: print(f"Resuming: {len(done)} variant(s) done.")
    print(f"\n{title}: {len(variants)} variants x {len(seeds)} seeds, "
          f"{cfg['ablation_epochs']} epochs (patience {cfg['ablation_early_stop_patience']})")
    if base_combo: print(f"  (on stage-1 branch winner: {base_combo})")
    for name, ov in variants.items():
        if name in done: print(f"{name:30s} [skipped]"); continue
        c = dict(cfg)
        if base_combo: c.update(base_combo)
        c.update(ov)
        c["epochs"] = cfg["ablation_epochs"]; c["early_stop_patience"] = cfg["ablation_early_stop_patience"]
        comp = complexity(build_model(c, num_classes), c["img_size"])
        val_accs, test_accs = [], []
        for sd in seeds:
            r = train_single(c, loaders, num_classes, sd, device)
            val_accs.append(r["best_val"]); test_accs.append(r["test_acc"])
        val_accs, test_accs = np.array(val_accs), np.array(test_accs)
        rows.append(dict(variant=name, val_mean=float(val_accs.mean()), val_std=_std(val_accs),
                         test_mean=float(test_accs.mean()), test_std=_std(test_accs),
                         params=comp["params"], FLOPs_M=round(comp["FLOPs_M"], 2)))
        json.dump(rows, open(out_path, "w"), indent=2)
        print(f"{name:30s} VAL={val_accs.mean()*100:6.2f}+/-{_std(val_accs)*100:4.2f}  "
              f"(test={test_accs.mean()*100:5.2f})  params={comp['params']:,}")
    rs = sorted(rows, key=lambda r: -r["val_mean"])
    print("\n  Ranked by VALIDATION (selection metric):")
    for r in rs:
        print(f"    {r['variant']:30s} val={r['val_mean']*100:6.2f}  params={r['params']:>7,d}")
    print(f"  -> winner (by val): {rs[0]['variant']}")
    print(f"Saved -> {out_path}")
    return rows


def run_block_design_ablation(cfg, loaders=None, base_combo=None):
    """Ablation 1: which coherent block design wins (val-ranked)."""
    return _val_ablation(cfg, BLOCK_DESIGN_VARIANTS, "ablation_block_design.json",
                         "ABLATION 1 -- BLOCK-DESIGN SEARCH", base_combo, loaders)


# ============= ABLATION 2: MULTIPLICITY + STRUCTURAL TOGGLES (on winner) =======
# Take the winning block design and ask "how many of it" (multiplicity, uniform
# depth n per stage) plus a few whole-unit structural toggles (drop residual,
# drop gate, max vs avg pool). NOT internal surgery -- whole-block presence and
# count. Multiplicity is not limited to 2/3/4; we include 1 (baseline) through 4.
def build_multiplicity_variants():
    v = {}
    for n in (1, 2, 3, 4):
        v[f"x{n} blocks/stage"] = dict(blocks=(n, n, n, n))
    # structural toggles at the winning multiplicity are added at runtime once
    # the winning n is known; here we also include fixed structural toggles at n=1
    v["x1, no residual"] = dict(blocks=(1, 1, 1, 1), use_residual=False)
    v["x1, max pool"]    = dict(blocks=(1, 1, 1, 1), pool_type="max")
    return v

MULTIPLICITY_VARIANTS = build_multiplicity_variants()


def run_multiplicity_ablation(cfg, loaders=None, base_combo=None, base_design=None):
    """Ablation 2: multiplicity + structural toggles, on the winning block design.
    base_combo = stage-1 branch winner; base_design = Ablation-1 block winner."""
    base = {}
    if base_combo: base.update(base_combo)
    if base_design: base.update(base_design)
    return _val_ablation(cfg, MULTIPLICITY_VARIANTS, "ablation_multiplicity.json",
                         "ABLATION 2 -- MULTIPLICITY + STRUCTURAL TOGGLES",
                         base if base else None, loaders)


DEPTH_VARIANTS = {f"depth={b}": dict(blocks=b) for b in
                  [(1,1,1,1),(2,1,1,1),(1,2,1,1),(1,1,2,1),(2,2,1,1),(1,2,2,1),(2,2,2,2)]}
WIDTH_VARIANTS = {f"width={w}": dict(channels=w) for w in
                  [(8,16,24,32),(12,24,32,40),(16,24,32,40),(16,32,40,48),(20,32,48,56)]}

# DEPTH_VARIANTS above varies blocks-per-stage non-uniformly, which conflates
# "how many ADA blocks total" with "which stage gets them" -- it can't isolate
# whether stacking depth itself helps, only where extra capacity is best placed.
# A naive UNIFORM depth sweep (n,n,n,n) at the default width also confounds
# depth with parameter count: n=2 uniformly costs 127K, already over budget.
# BUDGET_VARIANTS below holds total params roughly FIXED (~73-75K, matching the
# n=1 baseline) and trades width for depth, so depth-vs-width is an isolated,
# apples-to-apples comparison instead of "more capacity helped, somehow."
BUDGET_VARIANTS = {
    "n=1 blocks, wide  (baseline)": dict(blocks=(1,1,1,1), channels=(12,24,32,40)),
    "n=2 blocks, narrow (budget-matched)": dict(blocks=(2,2,2,2), channels=(8,16,24,32)),
    "n=3 blocks, narrower (budget-matched)": dict(blocks=(3,3,3,3), channels=(6,12,20,24)),
    "n=4 blocks, narrowest (budget-matched)": dict(blocks=(4,4,4,4), channels=(6,12,16,20)),
}


def run_budget_ablation(cfg, loaders=None, seeds=None):
    """Isolated depth-vs-width experiment at a roughly FIXED parameter budget
    (~73-75K, matching the n=1 baseline). Answers: for a fixed budget, is it
    better to go narrow-and-deep (more ADA blocks, fewer channels) or
    wide-and-shallow (current default)? Same checkpoint/resume pattern as the
    other ablations."""
    seeds = seeds or cfg["ablation_seeds"]
    device = "cuda" if torch.cuda.is_available() else "cpu"
    if loaders is None: loaders = build_loaders(cfg)
    num_classes = len(loaders[3])
    out_path = os.path.join(cfg["out_dir"], "ablation_budget.json")
    rows = json.load(open(out_path)) if os.path.exists(out_path) else []
    done = {r["variant"] for r in rows}
    if done: print(f"Resuming: {len(done)} variant(s) done.")
    print(f"\nBUDGET-MATCHED DEPTH-VS-WIDTH ABLATION: {len(BUDGET_VARIANTS)} x {len(seeds)} seeds, "
          f"{cfg['ablation_epochs']} epochs")
    for name, ov in BUDGET_VARIANTS.items():
        if name in done: print(f"{name:40s} [skipped]"); continue
        c = dict(cfg); c.update(ov)
        c["epochs"] = cfg["ablation_epochs"]; c["early_stop_patience"] = cfg["ablation_early_stop_patience"]
        comp = complexity(build_model(c, num_classes), c["img_size"])
        accs = np.array([train_single(c, loaders, num_classes, sd, device)["test_acc"] for sd in seeds])
        rows.append(dict(variant=name, acc_mean=float(accs.mean()), acc_std=_std(accs),
                         params=comp["params"], FLOPs_M=round(comp["FLOPs_M"], 2)))
        json.dump(rows, open(out_path, "w"), indent=2)
        print(f"{name:40s} acc={accs.mean()*100:6.2f}+/-{_std(accs)*100:4.2f}  "
              f"params={comp['params']:>7,d}  FLOPs={comp['FLOPs_M']:6.1f}M")
    print(f"Saved -> {out_path}")
    return rows


def run_depth_width_ablation(cfg, loaders=None, budget=100_000):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    if loaders is None: loaders = build_loaders(cfg)
    num_classes = len(loaders[3]); seeds = cfg["ablation_seeds"]
    out_path = os.path.join(cfg["out_dir"], "ablation_depth_width.json")
    saved = json.load(open(out_path)) if os.path.exists(out_path) else {}
    depth_rows, width_rows = saved.get("depth", []), saved.get("width", [])
    def ck(): json.dump(dict(depth=depth_rows, width=width_rows), open(out_path, "w"), indent=2)
    def sweep(variants, rows, title):
        done = {r["variant"] for r in rows}; print("\n" + title)
        for name, ov in variants.items():
            if name in done: print(f"{name:24s} [skipped]"); continue
            c = dict(cfg); c.update(ov)
            c["epochs"] = cfg["ablation_epochs"]; c["early_stop_patience"] = cfg["ablation_early_stop_patience"]
            comp = complexity(build_model(c, num_classes), c["img_size"]); over = comp["params"] > budget
            accs = np.array([train_single(c, loaders, num_classes, sd, device)["test_acc"] for sd in seeds])
            rows.append(dict(variant=name, acc_mean=float(accs.mean()),
                             acc_std=_std(accs),
                             params=comp["params"], FLOPs_M=round(comp["FLOPs_M"], 2), under_100k=not over))
            ck(); print(f"{name:24s} acc={accs.mean()*100:6.2f}+/-{_std(accs)*100:4.2f}  "
                        f"params={comp['params']:>7,d}{'  [OVER 100K]' if over else ''}")
    sweep(DEPTH_VARIANTS, depth_rows, "DEPTH sweep"); sweep(WIDTH_VARIANTS, width_rows, "WIDTH sweep")
    print(f"Saved -> {out_path}")
    return depth_rows, width_rows


def run_all(cfg):
    """One orchestrator: build the leak-free split ONCE, then run every phase
    in sequence -- main training, SNR stress ablation, component ablation,
    depth/width ablation. Each phase already checkpoints internally (per
    seed for training, per variant for the ablations), so if Kaggle kills
    the session partway through, simply re-running MODE='all' picks up
    exactly where it left off -- already-completed work is detected and
    skipped, not redone. No manual phase-selection needed across reruns."""
    t0 = time.time()
    print("=" * 88)
    print("RUN ALL: building the shared leak-free split once for every phase below")
    print("=" * 88)
    loaders = build_loaders(cfg)

    print("\n" + "#" * 88)
    print("# PHASE 1/5 -- MAIN TRAINING (multi-seed, full epoch budget)")
    print("#" * 88)
    run_experiment(cfg, loaders=loaders)

    print("\n" + "#" * 88)
    print(f"# PHASE 2/5 -- SNR-ROBUSTNESS STRESS ABLATION ({cfg['ablation_epochs']} epochs)")
    print("#" * 88)
    run_stress_ablation(cfg, loaders=loaders)

    print("\n" + "#" * 88)
    print(f"# PHASE 3/5 -- COMPONENT ABLATION ({cfg['ablation_epochs']} epochs)")
    print("#" * 88)
    run_component_ablation(cfg, loaders=loaders)

    print("\n" + "#" * 88)
    print(f"# PHASE 4/5 -- DEPTH/WIDTH ABLATION ({cfg['ablation_epochs']} epochs)")
    print("#" * 88)
    run_depth_width_ablation(cfg, loaders=loaders)

    print("\n" + "#" * 88)
    print(f"# PHASE 5/5 -- BUDGET-MATCHED DEPTH-VS-WIDTH ABLATION ({cfg['ablation_epochs']} epochs)")
    print("#" * 88)
    run_budget_ablation(cfg, loaders=loaders)

    hrs = (time.time() - t0) / 3600
    print("\n" + "=" * 88)
    print(f"ALL PHASES COMPLETE. Wall time this invocation: {hrs:.2f} hours.")
    print(f"Outputs in {cfg['out_dir']}/: ramdnet_summary.json, stress_ablation.json, "
          f"ablation_components.json, ablation_depth_width.json, ablation_budget.json")
    print("=" * 88)


if __name__ == "__main__":
    # ---- NO command-line args. Edit these, then run. ----
    CONFIG["data_dir"] = "/kaggle/input/datasets/kaushikar/drone-usat/DIAT-uSAT_dataset"  # <- set
    MODE = "train"   # "all" | "train" | "stress" | "ablation_combo" | "ablation_components" | "ablation_depth_width" | "ablation_budget" | "split_only"

    if MODE == "all":
        run_all(CONFIG)
    elif MODE == "train":
        run_experiment(CONFIG)
    elif MODE == "stress":
        run_stress_ablation(CONFIG)
    elif MODE == "ablation_combo":
        run_combo_ablation(CONFIG)
    elif MODE == "ablation_block_design":
        run_block_design_ablation(CONFIG)
    elif MODE == "ablation_multiplicity":
        run_multiplicity_ablation(CONFIG)
    elif MODE == "ablation_components":
        run_component_ablation(CONFIG)
    elif MODE == "ablation_depth_width":
        run_depth_width_ablation(CONFIG)
    elif MODE == "ablation_budget":
        run_budget_ablation(CONFIG)
    elif MODE == "split_only":
        loaders = build_loaders(CONFIG)
        print(f"\nSplit built. classes: {loaders[3]}")
    else:
        raise ValueError(f"Unknown MODE: {MODE}")

Device: cuda
Found 4849 images across 6 classes: ['3_long_blade_rotor', '3_short_blade_rotor', 'Bird', 'Bird+mini-helicopter', 'RC_plane', 'drone']
  3_long_blade_rotor      : 799
  3_short_blade_rotor     : 800
  Bird                    : 800
  Bird+mini-helicopter    : 815
  RC_plane                : 800
  drone                   : 835
Recording structure (by frame-index gaps, per raw sub-folder):
  3_long_blade_rotor      :   2 recording(s) across raw folder(s) ['3_long_blade_rotor'], sizes min/med/max = 353/399/446
  3_short_blade_rotor     :   2 recording(s) across raw folder(s) ['3_short_blade_rotor_1', '3_short_blade_rotor_2'], sizes min/med/max = 400/400/400
  Bird                    :   1 recording(s) across raw folder(s) ['Bird'], sizes min/med/max = 800/800/800
  Bird+mini-helicopter    :   2 recording(s) across raw folder(s) ['Bird+mini-helicopter_1', 'Bird+mini-helicopter_2'], sizes min/med/max = 400/407/415
  RC_plane                :   2 recording(s) across raw folder(s)